# 03: Feature Engineering, Non-Linear Transforms & Selection

**Track 02: Data Analytics, EDA & High-Performance Dataframes** | *Tensorbox AI/ML Production Curriculum*

---
### Overview & Objectives
Transform raw features into predictive signals: Target Encoding with smoothing, Polynomial combinations, Box-Cox/Yeo-Johnson power transforms, Mutual Information, and RFECV selection.


## 1. Ingest Housing Prices Dataset & Engineer Ratios
Construct domain-specific interaction features: Price per sqft, bath-to-bed ratio, age.

In [ ]:
import os
import sys
from pathlib import Path

for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (p / "utils").exists():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        break

from utils.data_loader import load_dataset
import pandas as pd
import numpy as np
from sklearn.preprocessing import PowerTransformer, StandardScaler
from sklearn.feature_selection import mutual_info_regression

df_housing = load_dataset("housing_prices", split="train")
print(f"Loaded Housing Dataset: {df_housing.shape}")

numeric_cols = df_housing.select_dtypes(include=[np.number]).columns.tolist()
price_col = "Price" if "Price" in df_housing.columns else ("medv" if "medv" in df_housing.columns else numeric_cols[-1])
unit_col = "SquareFeet" if "SquareFeet" in df_housing.columns else ("rm" if "rm" in df_housing.columns else numeric_cols[0])

# Feature Engineering
df_housing["Price_Per_Unit"] = df_housing[price_col] / (df_housing[unit_col] + 1)
if "YearBuilt" in df_housing.columns:
    df_housing["Age"] = 2026 - df_housing["YearBuilt"]

print(df_housing[[price_col, unit_col, "Price_Per_Unit"]].head())


## 2. Power Transformations for Skewed Targets & Features
Applying Yeo-Johnson transformation to stabilize variance.

In [ ]:
pt = PowerTransformer(method="yeo-johnson")
df_housing["Price_Transformed"] = pt.fit_transform(df_housing[[price_col]])

print(f"Original Target Skewness    : {df_housing[price_col].skew():.4f}")
print(f"Transformed Target Skewness : {df_housing['Price_Transformed'].skew():.4f}")


## 3. Mutual Information Feature Importance
Rank all continuous and engineered features according to their non-linear mutual information with Target Price.

In [ ]:
numeric_features = df_housing.select_dtypes(include=[np.number]).drop(columns=[price_col, "Price_Transformed"], errors="ignore").fillna(0)
y = df_housing[price_col]

mi_scores = mutual_info_regression(numeric_features, y, random_state=42)
mi_df = pd.DataFrame({"Feature": numeric_features.columns, "Mutual_Info_Score": mi_scores}).sort_values(by="Mutual_Info_Score", ascending=False)
print("=== Mutual Information Feature Ranking ===")
print(mi_df.head(8).to_string(index=False))
